In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. PREPARE DATA (CIFAR-10 + Text)
# ==========================================
print("  Loading CIFAR-10 and generating captions...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize images
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Generate Class-Specific Text
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

def generate_captions(labels):
    return np.array([f"A photo of a {class_names[l[0]]}" for l in labels])

train_texts = generate_captions(y_train)
test_texts = generate_captions(y_test)

# Vectorize Text
vectorizer = layers.TextVectorization(max_tokens=200, output_sequence_length=15)
vectorizer.adapt(train_texts)

x_train_text = vectorizer(train_texts)
x_test_text = vectorizer(test_texts)

# Create TensorFlow Dataset (Batching is crucial for Contrastive Learning)
BATCH_SIZE = 64
train_dataset = tf.data.Dataset.from_tensor_slices(((x_train, x_train_text))).shuffle(5000).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices(((x_test, x_test_text))).batch(BATCH_SIZE)

print(f"Data Ready. Batch Size: {BATCH_SIZE}")

# ==========================================
# 2. DEFINE THE TWO ENCODERS
# ==========================================
EMBEDDING_DIM = 64 # The size of the "Joint Space"

# --- Image Encoder ---
def create_image_encoder():
    inputs = Input(shape=(32, 32, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    # Projection Head (Standard in Contrastive Learning)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(EMBEDDING_DIM)(x) # No activation, raw vectors
    return models.Model(inputs, outputs, name="Image_Encoder")

# --- Text Encoder ---
def create_text_encoder():
    inputs = Input(shape=(15,))
    x = layers.Embedding(200, 32)(inputs)
    x = layers.GlobalAveragePooling1D()(x)
    # Projection Head
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(EMBEDDING_DIM)(x)
    return models.Model(inputs, outputs, name="Text_Encoder")

image_encoder = create_image_encoder()
text_encoder = create_text_encoder()

# ==========================================
# 3. DEFINE CONTRASTIVE LOSS (The Magic)
# ==========================================
class ContrastiveModel(models.Model):
    def __init__(self, image_encoder, text_encoder, temp=0.1):
        super().__init__()
        self.image_encoder = image_encoder
        self.text_encoder = text_encoder
        self.temp = temp # Temperature parameter to scale logits
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")

    def call(self, inputs):
        # This is used for inference
        imgs, txts = inputs
        img_feats = self.image_encoder(imgs)
        txt_feats = self.text_encoder(txts)
        return img_feats, txt_feats

    def train_step(self, data):
        # Unpack Data
        images, texts = data

        with tf.GradientTape() as tape:
            # 1. Forward Pass: Get features for the whole batch
            # Shape: (Batch_Size, 64)
            img_feats = self.image_encoder(images)
            txt_feats = self.text_encoder(texts)

            # 2. Normalize features (Cosine Similarity requires normalized vectors)
            img_feats = tf.math.l2_normalize(img_feats, axis=1)
            txt_feats = tf.math.l2_normalize(txt_feats, axis=1)

            # 3. Compute Similarity Matrix (Logits)
            # Matrix multiplication: (B, 64) x (64, B) -> (B, B)
            # This gives similarity of every image to every text in the batch
            logits = tf.matmul(img_feats, txt_feats, transpose_b=True)
            logits = logits / self.temp # Scale by temperature

            # 4. Compute Loss
            # The "labels" are just the indices [0, 1, 2, ... Batch_Size]
            # Because Image 0 should match Text 0, Image 1 match Text 1, etc.
            batch_size = tf.shape(logits)[0]
            labels = tf.range(batch_size)

            # We calculate loss in both directions: Image->Text and Text->Image
            loss_i2t = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
            loss_t2i = tf.keras.losses.sparse_categorical_crossentropy(labels, tf.transpose(logits), from_logits=True)

            loss = (loss_i2t + loss_t2i) / 2.0

        # 5. Backward Pass
        gradients = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))

        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

# ==========================================
# 4. TRAIN THE MODEL
# ==========================================
# Connect the encoders into the contrastive wrapper
clip_model = ContrastiveModel(image_encoder, text_encoder, temp=0.1)
clip_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

print("\nStarting Contrastive Training ...")
# Note: We don't provide 'y' labels because the model generates them internally (0, 1, 2...)
clip_model.fit(train_dataset, epochs=5)

# ==========================================
# 5. TEST: ZERO-SHOT RETRIEVAL
# ==========================================
print("\n--- Testing Joint Space Retrieval ---")

# Let's take 5 random images from the test set
sample_indices = np.random.choice(len(x_test), 5, replace=False)
sample_imgs = x_test[sample_indices]
true_labels = [class_names[y_test[i][0]] for i in sample_indices]

# Let's create a "Query Bank" of text descriptions for all 10 classes
queries = [f"A photo of a {c}" for c in class_names]
queries_vec = vectorizer(np.array(queries))

# Get Embeddings
img_embeddings = image_encoder.predict(sample_imgs, verbose=0)
txt_embeddings = text_encoder.predict(queries_vec, verbose=0)

# Normalize for Cosine Similarity
img_embeddings = img_embeddings / np.linalg.norm(img_embeddings, axis=1, keepdims=True)
txt_embeddings = txt_embeddings / np.linalg.norm(txt_embeddings, axis=1, keepdims=True)

# Calculate Similarity (Dot Product)
# (5 Images, 64) x (10 Texts, 64).T -> (5, 10) Matrix
similarity = np.dot(img_embeddings, txt_embeddings.T)

# Show Results
for i in range(5):
    # Find best match
    best_idx = np.argmax(similarity[i])
    predicted_label = class_names[best_idx]
    confidence = similarity[i][best_idx]

    print(f"Image True: {true_labels[i]:<12} | Pred: {predicted_label:<12} | Conf: {confidence:.2f}")
